# Chapter 8: The practical formula

*Code cells: MIT licence (`LICENSE`). Text and figures: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang, 2026).*

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'tutorial' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
pd.set_option('display.precision', 8)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
REF = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
LAM, F = REF['lambda'].to_numpy(), REF['F_continuum'].to_numpy()
Z_M, B_COEF, A_GAUSS = 0.71821164, 0.64680, 4/math.sqrt(3*math.pi)
def check(name, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.10g}  (expected {expected:.10g}, tol {tol:g})')
    return ok

## Construction

With $w=\sqrt\lambda$,
$$F_R=\frac{w^2}2\,\frac{1+a_Rw+b_Rw^2+c_Rw^3}{1+a_Rw+d_Rw^2+e_Rw^3+f_Rw^4}.$$
Six coefficients, four constraints: the rod limit is built in; $d_R-b_R=37/384$ enforces the coefficient
$37/768$; and the large-$w$ expansion must reproduce $4/\sqrt{3\pi}$, $-2z_M$ and $B$. The remaining two
parameters were fitted by minimax to 34 calibration nodes (`code/fit_correlation.py`); the other 33 nodes,
including the nine above $\lambda=5000$, were withheld.

In [2]:
coef = json.load(open(ROOT/'data'/'rational_fit_coefficients.json'))['central']; print(coef)
w = sp.symbols('w', positive=True)
a_, b_, c_, d_, e_, f_ = [sp.nsimplify(round(coef[k], 8), rational=True) for k in 'abcdef']
FR = w**2/2*(1+a_*w+b_*w**2+c_*w**3)/(1+a_*w+d_*w**2+e_*w**3+f_*w**4)
print('short:', sp.N(sp.series(FR, w, 0, 5).removeO(), 8), '   (37/768 =', float(sp.Rational(37,768)), ')')
print('d_R - b_R =', float(d_-b_), '   37/384 =', 37/384)
v = sp.symbols('v', positive=True)
longexp = sp.series(FR.subs(w, 1/v), v, 0, 2).removeO()
print('long :', sp.N(longexp, 8), '   expected 4/sqrt(3 pi) =', round(A_GAUSS, 8), ', -2 z_M =', -2*Z_M, ', B =', B_COEF)

{'a': 0.8235594197640312, 'b': 0.49501999959486265, 'c': 0.2388914844829849, 'd': 0.5913741662615293, 'e': 0.29102850317198314, 'f': 0.09167401363965846, 'B': 0.6468}


short: -0.048177085*w**4 + 0.5*w**2    (37/768 = 0.048177083333333336 )
d_R - b_R = 0.09635417    37/384 = 0.09635416666666667
long : 0.64680005*v - 1.4364234 + 1.3029401/v    expected 4/sqrt(3 pi) = 1.30294003 , -2 z_M = -1.43642328 , B = 0.6468


## Sign audit

Treating the printed coefficients as exact rationals, the numerators of $F_R'$, $-F_R''$,
$-d(F_R/\lambda)/d\lambda$ and $\lambda/2-F_R$ are polynomials in $w$ whose signs can be certified (`code/rational_symbolic_audit.py`).
The formula is therefore increasing, concave, has nonincreasing $F_R/\lambda$ and obeys $F_R<\lambda/2$ for
all $\lambda>0$, like the exact $F$.

In [3]:
from rational_symbolic_audit import audit_variant
rows, summary = audit_variant(coef, 8)
pd.DataFrame(summary)

,quantity,degree,nonzero_coefficients,minimum_sign,maximum_sign,denominator
0,F_prime,7,8,1,1,2*(9167401*w**4 + 29102850*w**3 + 59137417*w**...
1,minus_F_second,9,10,1,1,4*(9167401*w**4 + 29102850*w**3 + 59137417*w**...
2,minus_d_F_over_lambda,5,6,1,1,2*(9167401*w**4 + 29102850*w**3 + 59137417*w**...
3,rod_gap,6,3,1,1,2*(9167401*w**4 + 29102850*w**3 + 59137417*w**...


## Error at every node (Fig. 4)

In [4]:
import plot_main_figures as pmf
err = 100*(pmf.rational(LAM)/F - 1)
split = pd.read_csv(ROOT/'data'/'reference_nodes_with_split.csv').set_index('lambda')['set']
sets = pd.Series(LAM).map(split).fillna('holdout (added)')
for name, idx in pd.Series(LAM).groupby(sets.values).groups.items():
    print(f'{name:16s} n={len(idx):2d}  max |error| = {np.abs(err[list(idx)]).max():.4f} %')
check('max error on all 67 nodes (%)', np.abs(err).max(), 0.0924, 5e-4)
fig = pmf.figure4(); fig

calibration      n=34  max |error| = 0.0887 %
extrapolation    n= 9  max |error| = 0.0001 %
holdout          n=24  max |error| = 0.0924 %
PASS  max error on all 67 nodes (%): 0.09235706626  (expected 0.0924, tol 0.0005)


<Figure size 744.882x277.165 with 2 Axes>